# Phase 3 — Sequential Updating and Shrinkage

Companion notebook to `notes/phase3-sequential-updating.md`. We:

1. Demonstrate sequential = batch numerically across all four pairs.
2. Walk the FYF reference scenario month by month, plotting μₙ ± 1.96σₙ.
3. Reproduce the shrinkage table from §2.3 of the notes.
4. Show three disagreeing priors converging.
5. Verify the recursive (Kalman-gain) form.

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from src.conjugate import (
    NormalNormalUpdater,
    GammaPoissonUpdater,
    BetaBinomialUpdater,
    NormalInverseGammaUpdater,
)
from src.updating import (
    SequentialUpdater,
    normal_normal_shrinkage_weight,
    months_to_data_weight,
)

rng = np.random.default_rng(seed=20260603)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## 1. Sequential = Batch — numerical demonstration

Theorem (Phase 3 §1): feeding observations one at a time gives the
same posterior as feeding the whole batch in one go. We verify for
all four conjugate pairs.

In [ ]:
# Normal-Normal
nn_prior = NormalNormalUpdater(mu0=1_050_000, sigma0_sq=150_000**2, sigma_sq=80_000**2)
nn_data = list(rng.normal(loc=1_080_000, scale=80_000, size=12))
nn_seq = SequentialUpdater(nn_prior).feed_batch(nn_data)
nn_bat = nn_prior.update(nn_data)
print(f"Normal-Normal:    seq μ={nn_seq.mean():,.2f}, σ²={nn_seq.variance():.4e}")
print(f"                  bat μ={nn_bat.mean():,.2f}, σ²={nn_bat.variance():.4e}")
print(f"                  diff μ={abs(nn_seq.mean()-nn_bat.mean()):.2e}")

# Gamma-Poisson
gp_prior = GammaPoissonUpdater(alpha0=3.0, beta0=1.0)
gp_data = list(rng.poisson(lam=2.5, size=12))
gp_seq = SequentialUpdater(gp_prior).feed_batch(gp_data)
gp_bat = gp_prior.update(gp_data)
print(f"\nGamma-Poisson:    seq α={gp_seq.alpha}, β={gp_seq.beta}")
print(f"                  bat α={gp_bat.alpha}, β={gp_bat.beta}")

# Beta-Binomial
bb_prior = BetaBinomialUpdater(alpha0=2.0, beta0=8.0)
bb_data = [(int(rng.integers(0, 51)), 50) for _ in range(6)]
bb_seq = SequentialUpdater(bb_prior).feed_batch(bb_data)
bb_bat = bb_prior.update(
    successes=[s for s,_ in bb_data], trials=[t for _,t in bb_data]
)
print(f"\nBeta-Binomial:    seq α={bb_seq.alpha}, β={bb_seq.beta}")
print(f"                  bat α={bb_bat.alpha}, β={bb_bat.beta}")

# NIG
nig_prior = NormalInverseGammaUpdater(mu0=1_050_000, kappa0=1.0, alpha0=2.0, beta0=6.4e9)
nig_seq = SequentialUpdater(nig_prior).feed_batch(nn_data)
nig_bat = nig_prior.update(nn_data)
print(f"\nNIG:              seq μ={nig_seq.mu:,.2f}, β={nig_seq.beta:.4e}")
print(f"                  bat μ={nig_bat.mu:,.2f}, β={nig_bat.beta:.4e}")

## 2. Posterior trajectory across the FYF cycle

Reference parameters; 12 simulated months from N(1.08M, 80K). Plot μₙ
and the 95 % credible interval at each month.

In [ ]:
mu0, sigma0, sigma = 1_050_000.0, 150_000.0, 80_000.0
theta_star = 1_080_000.0

rng2 = np.random.default_rng(seed=20260601)
actuals = rng2.normal(loc=theta_star, scale=sigma, size=12)

prior = NormalNormalUpdater(mu0=mu0, sigma0_sq=sigma0**2, sigma_sq=sigma**2)
seq = SequentialUpdater(prior)
seq.feed_batch(list(actuals))

months = np.arange(1, 13)
means = np.array([e.posterior.mean() for e in seq.history()])
stds = np.array([e.posterior.std() for e in seq.history()])
weights = np.array(seq.shrinkage_weights())

print(f"{'n':>2} | {'μ_n':>13} | {'σ_n':>10} | {'95% CI low':>13} | {'high':>13} | w₀")
print("-" * 80)
for n, m, s, w in zip(months, means, stds, weights):
    lo, hi = m - 1.96*s, m + 1.96*s
    print(f"{n:>2} | R$ {m:>10,.0f} | R$ {s:>7,.0f} | R$ {lo:>10,.0f} | R$ {hi:>10,.0f} | {w:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.fill_between(months, means - 1.96*stds, means + 1.96*stds, alpha=0.20, color="crimson",
                label="95% credible interval")
ax.plot(months, means, "o-", color="crimson", lw=2, label=r"Posterior mean $\mu_n$")
ax.axhline(mu0, color="steelblue", ls=":", label=f"Prior μ₀={mu0:,.0f}")
ax.axhline(theta_star, color="black", ls="--", alpha=0.5, label=f"True θ★={theta_star:,.0f}")
ax.set_xlabel("Month n")
ax.set_ylabel(r"Mean monthly cost (R$)")
ax.set_title("Posterior trajectory and shrinking 95% CI across the FYF cycle")
ax.set_xticks(months)
ax.ticklabel_format(style="plain", axis="y")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 3. Shrinkage table (notes §2.3)

Closed-form $w_0(n)$ for the reference parameters. The data weight
first crosses 80 % at $n=2$ and 95 % at $n=6$.

In [ ]:
for n in [1, 2, 3, 6, 9, 12]:
    w = normal_normal_shrinkage_weight(sigma0, sigma, n)
    print(f"  n={n:>2}: prior weight = {w:.4f}, data weight = {1-w:.4f}")

for c in [0.80, 0.90, 0.95, 0.99]:
    n = months_to_data_weight(sigma0, sigma, c)
    print(f"  data weight ≥ {c:.0%} first at n = {n}")

## 4. Three disagreeing priors converge

Same data, three different priors. The disagreement decays at rate
$w_0(n)$ — exact theorem from §3.1 of the notes.

In [ ]:
priors_demo = {"low": 900_000, "reference": 1_050_000, "high": 1_200_000}
trajectories = {}
for name, m0 in priors_demo.items():
    p = NormalNormalUpdater(mu0=m0, sigma0_sq=sigma0**2, sigma_sq=sigma**2)
    s = SequentialUpdater(p)
    s.feed_batch(list(actuals))
    trajectories[name] = np.array([e.posterior.mean() for e in s.history()])

fig, ax = plt.subplots(figsize=(8, 4.2))
for name, traj in trajectories.items():
    ax.plot(months, traj, "o-", lw=2, label=f"{name}: μ₀={priors_demo[name]:,.0f}")
ax.axhline(theta_star, color="black", ls="--", alpha=0.5, label=f"θ★={theta_star:,.0f}")
ax.set_xlabel("Month n"); ax.set_ylabel(r"Posterior mean (R$)")
ax.set_title("Disagreeing priors converge to the same posterior")
ax.set_xticks(months); ax.ticklabel_format(style="plain", axis="y")
ax.legend(fontsize=9)
fig.tight_layout(); plt.show()

# Verify the gap shrinks at exactly w_0(n)
gap_obs = trajectories["high"] - trajectories["low"]
gap_pred = np.array([normal_normal_shrinkage_weight(sigma0, sigma, n) for n in months]) * (priors_demo["high"] - priors_demo["low"])
print("Gap comparison (should match exactly):")
for n, go, gp in zip(months, gap_obs, gap_pred):
    print(f"  n={n:>2}: observed = R$ {go:>9,.0f}, predicted w_0·gap_0 = R$ {gp:>9,.0f}")

## 5. Recursive Kalman-gain form

$\mu_n = \mu_{n-1} + K_n (x_n - \mu_{n-1})$, where
$K_n = \sigma^2_{n-1}/(\sigma^2_{n-1}+\sigma^2)$. Reproduce the engine's
trajectory using only the recursion, by hand.

In [ ]:
mu_manual, var_manual = mu0, sigma0**2
manual = []
for x in actuals:
    K = var_manual / (var_manual + sigma**2)
    mu_manual = mu_manual + K * (x - mu_manual)
    var_manual = var_manual * sigma**2 / (var_manual + sigma**2)
    manual.append(mu_manual)

engine = [e.posterior.mean() for e in seq.history()]
print(f"{'n':>2} | {'engine μ':>13} | {'manual μ':>13} | diff")
for n, e, m in zip(months, engine, manual):
    print(f"{n:>2} | R$ {e:>10,.2f} | R$ {m:>10,.2f} | {e-m:.2e}")

---

**Next phase.** `notes/phase4-predictive-inference.md` derives the
**posterior predictive distribution** — the answer to the question
the CFO actually wants: "what will next month / the year-end total
cost be?". The posterior on θ is a means; the predictive on $X_{t+1}$
is the end.